[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C26_Frontier_Agents_Course/03_computer_use/03_computer_use.ipynb)

# 03 · Computer Use（截图→动作循环）

目标：用**纯 numpy / 标准库**在一个玩具 GUI 里从零搭出 computer-use 的核心——**截图 → grounding → 动作解析 → 截图-动作循环 → 坐标映射 → 终止与护栏**，全程确定性、`assert` 验证、**无需 API key / 无需真实桌面**。

路线：玩具屏幕+元素 → grounding(意图→像素) → 动作解析 → MockGUIEnv 闭环 → 坐标映射 → 完整循环 → ✏️ 练习 → 📖 答案 → 🧪 真实 computer-use 动作形状胶囊。

> 心智模型：**computer use = agent 戴上人的眼睛(截图)和手(鼠标键盘)**。还是那张感知-动作循环，只是观察=截图、动作=点/打字。

## 1 · 玩具屏幕与 UI 元素（bounding box）

把「屏幕」建成一个 `H×W` 的 numpy 整数网格（每个元素填一个 id，0=空白）；每个可点的 UI 元素带一个 **bounding box** `(x1,y1,x2,y2)`。

`screenshot()` 返回当前屏幕数组——这就是 agent 每一步「看到」的东西。

In [ ]:
import numpy as np, json

SCREEN_W, SCREEN_H = 200, 120          # 真实屏幕尺寸(像素)

# 每个 UI 元素: name -> bounding box (x1, y1, x2, y2)，含左上、不含右下(半开区间)
ELEMENTS = {
    'login_btn':  (10, 10, 60, 30),    # 左上角一个按钮
    'user_field': (10, 50, 150, 70),   # 用户名输入框
    'submit_btn': (10, 90, 90, 110),   # 提交按钮
}
ELEM_ID = {name: i + 1 for i, name in enumerate(ELEMENTS)}   # 元素->像素填充值

def render(highlight=None):
    '''把元素画到屏幕网格上(用各自 id 填充其 box)，返回 H×W 数组=一张截图。'''
    screen = np.zeros((SCREEN_H, SCREEN_W), dtype=np.int16)
    for name, (x1, y1, x2, y2) in ELEMENTS.items():
        screen[y1:y2, x1:x2] = ELEM_ID[name]
    return screen

shot = render()
print('截图尺寸 (H, W):', shot.shape)
print('屏上出现的元素 id:', sorted(set(np.unique(shot)) - {0}))
# login_btn 的 box 区域应被填成它的 id
assert shot.shape == (SCREEN_H, SCREEN_W)
assert shot[15, 15] == ELEM_ID['login_btn']      # (x=15,y=15) 落在 login_btn 内
assert shot[0, 0] == 0                            # 角落是空白
assert set(np.unique(shot)) == {0, 1, 2, 3}       # 空白 + 3 个元素
print('✅ 玩具屏幕就绪：3 个元素各占一个 bounding box，screenshot() 返回像素网格')

## 2 · grounding：把意图落到像素（取中心点）

模型想「点 login_btn」，鼠标只认 `(x, y)`。**grounding** = 由元素名算出要点的像素。
最稳的目标是 box **中心点**（离边界最远，最能容忍误差）：`cx=(x1+x2)/2, cy=(y1+y2)/2`。

In [ ]:
def find_element(name):
    '''grounding: 元素名 -> 中心像素坐标 (cx, cy)，整数。'''
    if name not in ELEMENTS:
        raise KeyError(f'界面上没有元素: {name}')
    x1, y1, x2, y2 = ELEMENTS[name]
    return (x1 + x2) // 2, (y1 + y2) // 2

def hits(name, x, y):
    '''(x,y) 是否落在 name 的 bounding box 内(半开区间)。'''
    x1, y1, x2, y2 = ELEMENTS[name]
    return x1 <= x < x2 and y1 <= y < y2

for name in ELEMENTS:
    cx, cy = find_element(name)
    print(f'{name:11s} 中心=({cx:3d},{cy:3d})  命中自身={hits(name, cx, cy)}')
    assert hits(name, cx, cy), '点中心必须命中该元素'   # grounding 正确性
# 点中心命中的不只是『在框内』，还应是『正确的那个框』
cx, cy = find_element('submit_btn')
assert render()[cy, cx] == ELEM_ID['submit_btn']
# 未知元素要报错(挡住对不存在控件的 grounding)
try:
    find_element('ghost_btn'); raised = False
except KeyError:
    raised = True
assert raised
print('✅ grounding：意图->中心像素，点中心必命中正确元素；未知元素报错')

## 3 · 动作解析：把模型输出变成合法结构化动作

模型输出的是文本/字典(如 `{'action':'click','x':..,'y':..}`)，scaffold 必须**解析+校验**成可执行动作。
和模块 01 的工具调用解析一回事：识别动作类型(未知→报错)、校验参数(click 需数字 x/y、type 需 text)、并挡住越界坐标。

In [ ]:
VALID_ACTIONS = {'click', 'type', 'scroll', 'key', 'wait', 'screenshot', 'done'}

def parse_action(raw, screen_w=SCREEN_W, screen_h=SCREEN_H):
    '''解析+校验一个动作。返回规范化 action dict；非法则 raise ValueError。'''
    if not isinstance(raw, dict) or 'action' not in raw:
        raise ValueError('动作必须是含 action 字段的 dict')
    a = raw['action']
    if a not in VALID_ACTIONS:
        raise ValueError(f'未知动作: {a}')               # 挡住动作幻觉
    if a == 'click':
        x, y = raw.get('x'), raw.get('y')
        if not isinstance(x, int) or not isinstance(y, int):
            raise ValueError('click 需要整数 x, y')
        if not (0 <= x < screen_w and 0 <= y < screen_h):
            raise ValueError(f'坐标越界: ({x},{y}) 不在 {screen_w}x{screen_h} 内')
        return {'action': 'click', 'x': x, 'y': y}
    if a == 'type':
        if not isinstance(raw.get('text'), str):
            raise ValueError('type 需要字符串 text')
        return {'action': 'type', 'text': raw['text']}
    return {'action': a}                                  # wait/screenshot/done/key 等

print(parse_action({'action': 'click', 'x': 35, 'y': 20}))
print(parse_action({'action': 'type', 'text': 'alice'}))
print(parse_action({'action': 'done'}))
assert parse_action({'action': 'click', 'x': 35, 'y': 20})['x'] == 35
for bad in [{'action': 'fly'}, {'action': 'click', 'x': 1}, {'action': 'click', 'x': 999, 'y': 1},
            {'action': 'type'}, {'no_action': 1}]:
    try:
        parse_action(bad); ok = False
    except ValueError:
        ok = True
    assert ok, f'应拒绝非法动作: {bad}'
print('✅ 动作解析：合法放行；未知动作/缺参/类型错/越界坐标 全部拒绝')

## 4 · MockGUIEnv：确定性的 GUI 环境（截图-动作转移）

把环境实现成 `reset()` / `step(action) -> (screenshot, reward, done)`。
玩具任务 3 步：**点 login_btn → 在 user_field 输入 → 点 submit_btn**。只有按对顺序操作才推进状态，最后一步给 reward=1、done=True。完全确定。

In [ ]:
class MockGUIEnv:
    '''确定性 GUI: 登录任务。stage 0->1->2->done。
       正确动作序列: click(login)->type(...)->click(submit)。'''
    def __init__(self):
        self.reset()
    def reset(self):
        self.stage = 0
        self.typed = ''
        return self.screenshot()
    def screenshot(self):
        # 截图里多编码一行 stage 信息(末行像素=stage)，模拟『界面随状态变化』
        s = render().copy()
        s[-1, :] = self.stage
        return s
    def step(self, action):
        act = parse_action(action)
        reward, done = 0.0, False
        a = act['action']
        if self.stage == 0 and a == 'click' and hits('login_btn', act['x'], act['y']):
            self.stage = 1
        elif self.stage == 1 and a == 'type' and len(act['text']) > 0:
            self.typed = act['text']; self.stage = 2
        elif self.stage == 2 and a == 'click' and hits('submit_btn', act['x'], act['y']):
            self.stage = 3; reward, done = 1.0, True
        # 其它(顺序错/点空白)不推进 stage —— 状态不变
        return self.screenshot(), reward, done

env = MockGUIEnv()
obs = env.reset()
cx, cy = find_element('login_btn')
obs, r, done = env.step({'action': 'click', 'x': cx, 'y': cy}); print('点登录 -> stage', env.stage, 'r', r)
obs, r, done = env.step({'action': 'type', 'text': 'alice'});   print('输入   -> stage', env.stage, 'r', r)
cx, cy = find_element('submit_btn')
obs, r, done = env.step({'action': 'click', 'x': cx, 'y': cy}); print('点提交 -> stage', env.stage, 'r', r, 'done', done)
assert env.stage == 3 and r == 1.0 and done is True
# 顺序错(一上来就点提交)不应推进
env2 = MockGUIEnv()
_, r2, d2 = env2.step({'action': 'click', 'x': 50, 'y': 100})
assert env2.stage == 0 and r2 == 0.0 and d2 is False
print('✅ MockGUIEnv：按对顺序 3 步完成给 reward=1；顺序错则状态不变')

## 5 · 坐标映射：缩略图坐标 → 真实坐标

模型常看**缩小的截图**（省 token）。它在缩略图上给的坐标，必须乘以缩放因子换算回真实屏幕：
`x_real = x_shot * (W/w)`，`y_real = y_shot * (H/h)`。漏掉这步就点偏。

In [ ]:
def downscale(screen, shot_w, shot_h):
    '''把真实截图最近邻缩放到 (shot_h, shot_w)，模拟模型看到的缩略图。'''
    H, W = screen.shape
    ys = (np.arange(shot_h) * H // shot_h)
    xs = (np.arange(shot_w) * W // shot_w)
    return screen[np.ix_(ys, xs)]

def shot_to_real(xs, ys, shot_w, shot_h, real_w=SCREEN_W, real_h=SCREEN_H):
    '''缩略图坐标 -> 真实屏幕坐标(乘缩放因子)。'''
    x_real = int(xs * real_w / shot_w)
    y_real = int(ys * real_h / shot_h)
    return x_real, y_real

SHOT_W, SHOT_H = 50, 30                 # 模型看到的小缩略图
thumb = downscale(render(), SHOT_W, SHOT_H)
print('真实截图', render().shape, '-> 缩略图', thumb.shape)
# 真实中心 -> 缩到缩略图坐标 -> 再映射回真实，应仍命中同一元素(往返一致)
cx, cy = find_element('submit_btn')                         # 真实中心
xs = int(cx * SHOT_W / SCREEN_W); ys = int(cy * SHOT_H / SCREEN_H)   # 缩到缩略图
rx, ry = shot_to_real(xs, ys, SHOT_W, SHOT_H)               # 映射回真实
print(f'真实中心({cx},{cy}) -> 缩略图({xs},{ys}) -> 映射回({rx},{ry})')
assert hits('submit_btn', rx, ry), '映射回的坐标应仍落在 submit_btn 内'
# 若忘了缩放(直接拿缩略图坐标当真实坐标)，几乎必然点偏
assert not hits('submit_btn', xs, ys), '不换算 -> 点偏(印证缩放不可省)'
print('✅ 坐标映射：缩略图坐标 × 缩放因子 = 真实坐标；往返一致；漏换算则点偏')

## 6 · 完整截图-动作循环（接 MockLLM 策略）

把一切拼起来：一个**规则驱动的 MockLLM 策略**根据当前 stage 给出下一个动作，循环 `截图→动作→执行→新截图`，直到 `done` 或 `max_steps`。
**必须在 max_steps 内终止**——这是铁律。

In [ ]:
def mock_gui_policy(stage):
    '''确定性策略: 看 stage 决定下一动作(真实里是模型看截图决定)。'''
    if stage == 0:
        cx, cy = find_element('login_btn');  return {'action': 'click', 'x': cx, 'y': cy}
    if stage == 1:
        return {'action': 'type', 'text': 'alice'}
    if stage == 2:
        cx, cy = find_element('submit_btn'); return {'action': 'click', 'x': cx, 'y': cy}
    return {'action': 'done'}

def run_gui_agent(env, policy, max_steps=10):
    '''截图-动作循环。返回 (是否成功, 步数, 动作轨迹)。'''
    obs = env.reset()
    traj = []
    for step in range(max_steps):
        action = policy(env.stage)          # 真实里: policy(obs) 看截图
        traj.append(action)
        if action['action'] == 'done':
            return env.stage == 3, step, traj
        obs, reward, done = env.step(action)
        if done:
            return True, step + 1, traj
    return False, max_steps, traj           # 超步数也要停!

ok, steps, traj = run_gui_agent(MockGUIEnv(), mock_gui_policy, max_steps=10)
print(f'成功={ok}, 用了 {steps} 步')
for i, a in enumerate(traj):
    print(f'  step {i}: {a}')
assert ok is True and steps == 3, '应在 3 步内完成登录任务'
assert len(traj) <= 10, '必须在 max_steps 内终止'
print('✅ 完整截图-动作循环跑通：3 步完成任务且在 max_steps 内终止')

---
## ✏️ 练习 1：越界点击护栏（防误操作）

护栏的第一层是**坐标边界检查**：点屏幕外、或点禁区，都要拒绝。

实现 `is_in_bounds(x, y, w, h)`（坐标是否在 `[0,w)×[0,h)` 内）和 `guarded_click(x, y, w, h, forbidden)`：越界或落在 `forbidden`(一组 box) 内则 `raise PermissionError`，否则返回 `(x, y)`。

In [ ]:
def is_in_bounds(x, y, w, h):
    # TODO: 返回 (x,y) 是否在 [0,w) x [0,h) 内 (bool)
    raise NotImplementedError

def guarded_click(x, y, w, h, forbidden=()):
    # TODO: 1) 越界 -> raise PermissionError('坐标越界')
    #       2) 落在 forbidden 里任一 box (x1,y1,x2,y2, 半开) -> raise PermissionError('禁区')
    #       3) 否则返回 (x, y)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert is_in_bounds(5, 5, 200, 120) is True
assert is_in_bounds(-1, 5, 200, 120) is False and is_in_bounds(5, 999, 200, 120) is False
DANGER = [(0, 0, 20, 20)]            # 左上角是禁区(如关闭按钮)
assert guarded_click(100, 60, 200, 120, DANGER) == (100, 60)   # 安全点放行
for bx, by in [(-5, 10), (250, 10), (5, 5)]:    # 越界、越界、禁区
    try:
        guarded_click(bx, by, 200, 120, DANGER); blocked = False
    except PermissionError:
        blocked = True
    assert blocked, f'应拒绝危险点击 ({bx},{by})'
print('✅ 练习 1 通过：越界与禁区点击被护栏拦下，安全点放行')

## ✏️ 练习 2：卡住检测（防死循环）

GUI agent 常陷在「反复点同一无效按钮」里。若**最近 K 张截图完全相同**(界面没因动作变化)，应判定卡住、停止。

实现 `is_stuck(shots, k)`：`shots` 是按时间顺序的截图(numpy 数组)列表；若最后 `k` 张**两两相同**返回 `True`，否则 `False`(不足 k 张返回 False)。

In [ ]:
def is_stuck(shots, k=3):
    # TODO: 不足 k 张 -> False；
    #       取最后 k 张，若它们全部 np.array_equal 于第一张 -> True，否则 False
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
A = render(); B = render().copy(); B[0, 0] = 7      # A、B 不同
# 最近 3 张都是 A -> 卡住
assert is_stuck([B, A, A, A], k=3) is True
# 最近 3 张里有变化 -> 没卡
assert is_stuck([A, A, B], k=3) is False
# 不足 k 张 -> False
assert is_stuck([A, A], k=3) is False
print('✅ 练习 2 通过：连续 k 张截图相同判定卡住，可用于终止循环')

## ✏️ 练习 3：任意分辨率的坐标重映射

把第 5 节的映射推广到任意 `(shot_w, shot_h) -> (real_w, real_h)`。

实现 `remap(x, y, shot_w, shot_h, real_w, real_h)`：返回真实坐标 `(int(x*real_w/shot_w), int(y*real_h/shot_h))`。再用它验证一次「真实→缩略→映射回」的往返命中。

In [ ]:
def remap(x, y, shot_w, shot_h, real_w, real_h):
    # TODO: 返回 (int(x * real_w / shot_w), int(y * real_h / shot_h))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 100x100 缩略图上的 (25,25)，映射到 400x200 真实屏 -> (100,50)
assert remap(25, 25, 100, 100, 400, 200) == (100, 50)
# 往返: 真实点 -> 缩到 80x48 -> 映射回，应仍命中 user_field
cx, cy = find_element('user_field')
sw, sh = 80, 48
xs = int(cx * sw / SCREEN_W); ys = int(cy * sh / SCREEN_H)
rx, ry = remap(xs, ys, sw, sh, SCREEN_W, SCREEN_H)
assert hits('user_field', rx, ry), '往返映射应仍落在 user_field 内'
print('✅ 练习 3 通过：任意分辨率坐标重映射正确，往返命中')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def is_in_bounds(x, y, w, h):
    return 0 <= x < w and 0 <= y < h

def guarded_click(x, y, w, h, forbidden=()):
    if not is_in_bounds(x, y, w, h):
        raise PermissionError('坐标越界')
    for (x1, y1, x2, y2) in forbidden:
        if x1 <= x < x2 and y1 <= y < y2:
            raise PermissionError('禁区')
    return (x, y)

In [ ]:
# 练习 2 参考答案
def is_stuck(shots, k=3):
    if len(shots) < k:
        return False
    last = shots[-k:]
    return all(np.array_equal(s, last[0]) for s in last)

In [ ]:
# 练习 3 参考答案
def remap(x, y, shot_w, shot_h, real_w, real_h):
    return (int(x * real_w / shot_w), int(y * real_h / shot_h))

---
## 🧪 真实数据胶囊：真实 Anthropic computer-use 动作形状

Anthropic computer use 工具里，模型产出的动作是一组**确定形状**的指令。下面是几个**贴近真实**的动作（形如官方 computer use 工具的 action 字段），我们写一个转换器把本课的内部动作映射成它的形状，体会真实接口与本课的一一对应。

> 形状对照：真实 click 用 `{'action':'left_click','coordinate':[x,y]}`；输入用 `{'action':'type','text':...}`；快捷键 `{'action':'key','text':'ctrl+s'}`；截图 `{'action':'screenshot'}`。

In [ ]:
# 真实 Anthropic computer-use 动作样例(形状贴近官方工具)
REAL_ACTIONS = [
    {'action': 'screenshot'},
    {'action': 'left_click', 'coordinate': [35, 20]},
    {'action': 'type', 'text': 'alice'},
    {'action': 'key', 'text': 'Return'},
]

def to_anthropic(action):
    '''把本课内部动作 -> Anthropic computer-use 动作形状。'''
    a = action['action']
    if a == 'click':
        return {'action': 'left_click', 'coordinate': [action['x'], action['y']]}
    if a == 'type':
        return {'action': 'type', 'text': action['text']}
    if a == 'key':
        return {'action': 'key', 'text': action['combo']}
    return {'action': a}          # screenshot / wait 等同形

print('内部 click  ->', to_anthropic({'action': 'click', 'x': 35, 'y': 20}))
print('内部 type   ->', to_anthropic({'action': 'type', 'text': 'alice'}))
print('内部 screenshot ->', to_anthropic({'action': 'screenshot'}))
assert to_anthropic({'action': 'click', 'x': 35, 'y': 20}) == {'action': 'left_click', 'coordinate': [35, 20]}
assert to_anthropic({'action': 'screenshot'}) == {'action': 'screenshot'}
# 真实动作样例里的坐标确实落在我们的 login_btn 上(印证形状互通)
click_act = [a for a in REAL_ACTIONS if a['action'] == 'left_click'][0]
x, y = click_act['coordinate']
assert hits('login_btn', x, y)
print('✅ 本课内部动作可直接映射到真实 Anthropic computer-use 动作形状')

**🧪 胶囊练习**：实现 `count_clicks(trace)`：给定一串真实 computer-use 动作(如上 `REAL_ACTIONS`)，返回其中**点击类动作**(`left_click`/`right_click`/`double_click`)的数量。(审计一条动作轨迹「点了几次」是安全监控的常见操作。)

In [ ]:
def count_clicks(trace):
    # TODO: 返回 trace 中 action 在 {'left_click','right_click','double_click'} 里的条数
    raise NotImplementedError

In [ ]:
# 自测
trace = REAL_ACTIONS + [{'action': 'double_click', 'coordinate': [50, 100]}]
assert count_clicks(trace) == 2          # left_click + double_click
assert count_clicks([{'action': 'type', 'text': 'x'}]) == 0
print('点击类动作数:', count_clicks(trace))
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def count_clicks(trace):
    clicks = {'left_click', 'right_click', 'double_click'}
    return sum(1 for a in trace if a.get('action') in clicks)

---
## 🔧 旁注：对应的真实 Anthropic computer-use 调用

本课用 numpy 模拟的截图-动作循环，换成真实 Anthropic computer use 只是把『环境』换成真实桌面、把动作交给 `pyautogui`/`xdotool` 执行(伪代码，**本环境不跑、需 API key + 真实桌面**)：

```python
import anthropic, pyautogui
client = anthropic.Anthropic()                       # 读 ANTHROPIC_API_KEY
messages = [{'role': 'user', 'content': '帮我登录这个网站'}]
while True:
    resp = client.beta.messages.create(
        model='claude-opus-4-8', max_tokens=1024,
        tools=[{'type': 'computer_20250124', 'name': 'computer',
                'display_width_px': 1280, 'display_height_px': 800}],
        betas=['computer-use-2025-01-24'], messages=messages,
    )
    for block in resp.content:
        if block.type == 'tool_use':                 # 模型给出一个动作
            act = block.input                        # {'action':'left_click','coordinate':[x,y]} ...
            # 我们本地执行该动作(对应本课 env.step) —— 护栏在这里把关!
            if act['action'] == 'left_click':
                pyautogui.click(*act['coordinate'])
            shot = pyautogui.screenshot()            # 重新截图(对应本课 screenshot)
            # 把截图作为 tool_result 回传(base64 图像) -> 下一轮
    if resp.stop_reason == 'end_turn':
        break
```

对应关系：MockGUIEnv ↔ 真实桌面、`parse_action` ↔ 解析 `block.input`、`env.step` ↔ 本地执行动作 + 回传新截图、我们的 grounding / 坐标映射 / 护栏 / 终止逻辑**原样适用**。这就是「scaffold 可迁移」。

### 小结
- computer use = agent 戴上人的眼睛(截图)和手(鼠标键盘)，能操作**任意**软件(无需 API)。
- 还是感知-动作循环：观察=截图、动作=click/type/...，一次一个原子动作、边看边走。
- **grounding**(意图→像素)是核心瓶颈：点 bounding box 中心最稳。
- **坐标映射**：缩略图坐标 × 缩放因子 = 真实坐标，漏换算就点偏。
- **终止**多判据取或，max_steps 兜底 + 卡住检测，永远先保证『一定会停』。
- **护栏**：越界/禁区拒绝、高危动作需人确认、屏幕内容当数据而非命令(防注入)。

下一站：**模块 04 · Agentic RL** —— 用这个循环产生的轨迹 + 可验证奖励，反过来训练策略本身。